# Setup - Primera Exploración

Carga de los 9 CSV's de Olist a la capa 'bronze' de DuckDB, y exploración del dataset

In [1]:
import duckdb
from pathlib import Path

DATA_DIR = Path("../../Data") # carpeta con mis CSVs
DB_PATH = Path("../olist.duckdb") 

tablas = {
    "orders": "olist_orders_dataset",
    "order_items": "olist_order_items_dataset",
    "customers": "olist_customers_dataset",
    "products": "olist_products_dataset",
    "sellers": "olist_sellers_dataset",
    "payments": "olist_order_payments_dataset",
    "reviews": "olist_order_reviews_dataset",
    "geolocation": "olist_geolocation_dataset",
    "category_translation": "product_category_name_translation",
}

con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS bronze")

for tabla, archivo in tablas.items():
    csv = DATA_DIR / f"{archivo}.csv"
    assert csv.exists(), f"No encuentro el CSV: {csv.resolve()}"
    con.execute(f"""
        CREATE OR REPLACE TABLE bronze.{tabla} AS
        SELECT * FROM read_csv_auto('{csv.as_posix()}')
    """)

    n = con.execute(f"SELECT count(*) FROM bronze.{tabla}").fetchone()[0]
    print(f" bronze.{tabla:22s} <- {archivo}.csv({n:,} filas)")

con.close()
print("Carga completa.")

 bronze.orders                 <- olist_orders_dataset.csv(99,441 filas)
 bronze.order_items            <- olist_order_items_dataset.csv(112,650 filas)
 bronze.customers              <- olist_customers_dataset.csv(99,441 filas)
 bronze.products               <- olist_products_dataset.csv(32,951 filas)
 bronze.sellers                <- olist_sellers_dataset.csv(3,095 filas)
 bronze.payments               <- olist_order_payments_dataset.csv(103,886 filas)
 bronze.reviews                <- olist_order_reviews_dataset.csv(99,224 filas)
 bronze.geolocation            <- olist_geolocation_dataset.csv(1,000,163 filas)
 bronze.category_translation   <- product_category_name_translation.csv(71 filas)
Carga completa.


In [ ]:
con = duckdb.connect(str(DB_PATH))
print(con.execute("SELECT COUNT(*) FROM bronze.orders").fetchone()[0])
con.execute("SHOW TABLES").df()     # lista las tablas del bronze

99441


,name


## Conociendo nuestra Data:

1. Esquema y tipos de cada tabla:

Qué columnas tiene cada tabla y qué tipo les infirió `read_csv_auto`

In [3]:
con.execute("DESCRIBE bronze.orders").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,customer_id,VARCHAR,YES,None,None,None
2,order_status,VARCHAR,YES,None,None,None
3,order_purchase_timestamp,TIMESTAMP,YES,None,None,None
4,order_approved_at,TIMESTAMP,YES,None,None,None
5,order_delivered_carrier_date,TIMESTAMP,YES,None,None,None
6,order_delivered_customer_date,TIMESTAMP,YES,None,None,None
7,order_estimated_delivery_date,TIMESTAMP,YES,None,None,None


2. Rango de Fechas:

Ver de dónde a dónde va el dataset $\rightarrow$ definir el eje de las cohortes

In [4]:
con.execute("""
    SELECT min(order_purchase_timestamp), max(order_purchase_timestamp)
    FROM bronze.orders
""").df()

,min(order_purchase_timestamp),max(order_purchase_timestamp)
0,2016-09-04 21:15:19,2018-10-17 17:30:18


3. Revisar Nulls

Ver dónde faltan datos. Ej. en `orders` las fechas de entrega tienen nulls

In [5]:
con.execute("""
    SELECT
        count(*) AS total,
        count(order_delivered_customer_date) AS con_entrega,
        count(*) - count(order_delivered_customer_date) AS sin_entrega
    FROM bronze.orders
""").df()

,total,con_entrega,sin_entrega
0,99441,96476,2965


4. Revisando el id `customer_unique_id`

Observamos que hay dos diferentes `customer_id` y `customer_unique_id`

In [6]:
con.execute("""
    SELECT
        count(*) AS filas,
        count(DISTINCT customer_id) AS ids_por_orden,
        count(DISTINCT customer_unique_id) AS personas_reales
    FROM bronze.customers
""").df()

,filas,ids_por_orden,personas_reales
0,99441,99441,96096


5. Ver duplicados

Analizar las PKs. e.g. que `order_items` es único por (`order_id`, `order_item_id`) y que `orders.order_id` no tiene duplicados.

In [7]:
con.execute("""
    SELECT
        count(*) AS filas,
        count(DISTINCT order_id) AS ordenes_unicas
    FROM bronze.orders
""").df()

,filas,ordenes_unicas
0,99441,99441
